# 🌍 ImpactForge AI
## AI Multi-Agent Social Impact Project Designer

### AI for Good

ImpactForge AI is a multi-agent system powered by Google's Gemini models that transforms a social problem into a complete implementation-ready project proposal.

The workflow includes:

- Research Agent
- SDG Mapping Agent
- Planning Agent
- Metrics Agent
- Risk Assessment Agent
- Funding Agent
- Evaluation Agent
- Report Generation Agent

Each agent performs one specialized task and shares information through a centralized AgentState.

In [2]:
!pip -q install google-genai
!pip -q install pydantic

In [3]:
import json
import re

from typing import List, Optional

from pydantic import BaseModel

from google import genai

In [20]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

API_KEY = user_secrets.get_secret("GEMINI_API_KEY")

client = genai.Client(
    api_key=API_KEY
)

In [5]:
def parse_json(text):

    text = re.sub(r"```json", "", text)
    text = re.sub(r"```", "", text)

    text = text.strip()

    return json.loads(text)

In [6]:
def run_gemini(system_prompt, user_prompt):

    prompt = f"""
{system_prompt}

{user_prompt}
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return parse_json(response.text)

In [7]:
class ProjectIdea(BaseModel):

    problem: str
    location: str
    target_population: str
    budget: str

# Output Models

In [9]:
class ResearchOutput(BaseModel):
    summary: str
    root_causes: List[str]
    stakeholders: List[str]
    existing_solutions: List[str]
    challenges: List[str]


class SDG(BaseModel):
    number: int
    title: str
    reason: str


class SDGOutput(BaseModel):
    sdgs: List[SDG]


class PlanningOutput(BaseModel):
    objectives: List[str]
    activities: List[str]
    timeline: List[dict]


class MetricsOutput(BaseModel):
    kpis: List[str]
    success_metrics: List[str]


class Risk(BaseModel):
    risk: str
    impact: str
    probability: str
    mitigation: str


class RiskOutput(BaseModel):
    risks: List[Risk]


class FundingSource(BaseModel):
    organization: str
    funding_type: str
    reason: str
    estimated_amount: str


class FundingOutput(BaseModel):
    funding_sources: List[FundingSource]


class EvaluationOutput(BaseModel):
    innovation_score: int
    feasibility_score: int
    impact_score: int
    sustainability_score: int
    overall_score: int
    feedback: str


class ReportOutput(BaseModel):
    executive_summary: str
    implementation_strategy: str
    expected_impact: str
    recommendations: str

# Agent State 

In [10]:
class AgentState(BaseModel):

    project: ProjectIdea

    research: Optional[ResearchOutput] = None
    sdg: Optional[SDGOutput] = None
    planning: Optional[PlanningOutput] = None
    metrics: Optional[MetricsOutput] = None
    risks: Optional[RiskOutput] = None
    funding: Optional[FundingOutput] = None
    evaluation: Optional[EvaluationOutput] = None
    report: Optional[ReportOutput] = None

In [11]:
project = ProjectIdea(
    problem="Reduce food waste in Delhi schools",
    location="Delhi",
    target_population="School students",
    budget="50000 INR"
)

In [12]:
state = AgentState(
    project=project
)

state

AgentState(project=ProjectIdea(problem='Reduce food waste in Delhi schools', location='Delhi', target_population='School students', budget='50000 INR'), research=None, sdg=None, planning=None, metrics=None, risks=None, funding=None, evaluation=None, report=None)

In [13]:
RESEARCH_PROMPT = """
You are an expert social impact researcher.

Analyze the given social problem and identify:
1. A concise summary.
2. Root causes.
3. Key stakeholders.
4. Existing solutions.
5. Major implementation challenges.

Return ONLY valid JSON.

{
    "summary":"",
    "root_causes":[],
    "stakeholders":[],
    "existing_solutions":[],
    "challenges":[]
}
"""

In [14]:
def research_agent(state):

    user_prompt = f"""
Problem:
{state.project.problem}

Location:
{state.project.location}

Target Population:
{state.project.target_population}
"""

    data = run_gemini(
        RESEARCH_PROMPT,
        user_prompt
    )

    state.research = ResearchOutput(**data)

    return state

In [21]:
state = research_agent(state)

state.research

ResearchOutput(summary="Significant food waste is generated by school students in Delhi's educational institutions, leading to environmental strain, economic inefficiency, and loss of potential nutrition for those in need.", root_causes=['Lack of awareness among students and parents about the environmental and social impact of food waste.', 'Inaccurate portion sizing: Students often receive or are packed more food than they can or wish to consume.', 'Picky eating and dislike for certain food items, particularly vegetables, leading to uneaten portions.', 'Limited time allocated for lunch breaks, causing students to rush and discard unfinished meals.', 'Monotonous or unappealing menu options in school canteens, if provided, or lack of variety in packed lunches.', 'Absence of proper food waste management infrastructure within schools (e.g., composting facilities, clear segregation bins).', 'Social norms where discarding food is not widely stigmatized, or even seen as a sign of abundance.'

In [16]:
SDG_PROMPT = """
You are an expert in the United Nations Sustainable Development Goals.

Based on the problem and research findings, identify the most relevant SDGs.

Return ONLY valid JSON.

{
    "sdgs":[
        {
            "number":0,
            "title":"",
            "reason":""
        }
    ]
}
"""

In [17]:
def sdg_agent(state):

    user_prompt = f"""
Problem:
{state.project.problem}

Research Summary:
{state.research.summary}
"""

    data = run_gemini(
        SDG_PROMPT,
        user_prompt
    )

    state.sdg = SDGOutput(**data)

    return state

In [22]:
state = sdg_agent(state)

state.sdg

SDGOutput(sdgs=[SDG(number=12, title='Responsible Consumption and Production', reason='Reducing food waste directly aligns with SDG 12, particularly Target 12.3, which aims to halve per capita global food waste at the retail and consumer levels. The problem highlights economic inefficiency and environmental strain, both addressed by promoting sustainable consumption and production patterns.'), SDG(number=2, title='Zero Hunger', reason="The research findings explicitly mention the 'loss of potential nutrition for those in need.' Reducing food waste in schools can free up resources or make food available that could contribute to addressing hunger and improving food security, directly supporting SDG 2."), SDG(number=13, title='Climate Action', reason='Food waste is a significant contributor to greenhouse gas emissions, especially methane from landfills, and represents wasted energy and resources used in food production. Addressing food waste directly contributes to mitigating climate chan

In [38]:
PLAN_PROMPT = """
You are an experienced Project Manager.

Generate a project implementation plan.

IMPORTANT:

Return ONLY this JSON format.

{
    "objectives":[
        "Objective 1",
        "Objective 2"
    ],
    "activities":[
        "Activity 1",
        "Activity 2"
    ],
    "timeline":[
        {
            "Phase":"Month 1-2"
        }
    ]
}

Do NOT include IDs.
Do NOT include nested objects.
Do NOT include descriptions inside objects.
Return ONLY arrays of plain strings.
"""

In [25]:
def planning_agent(state):

    user_prompt = f"""
Problem:
{state.project.problem}

Research Summary:
{state.research.summary}

Relevant SDGs:
{", ".join([sdg.title for sdg in state.sdg.sdgs])}
"""

    data = run_gemini(
        PLAN_PROMPT,
        user_prompt
    )

    state.planning = PlanningOutput(**data)

    return state

In [26]:
METRICS_PROMPT = """
You are a Monitoring and Evaluation expert.

Based on the project objectives, create measurable KPIs and success metrics.

Return ONLY valid JSON.

{
    "kpis":[],
    "success_metrics":[]
}
"""

In [27]:
def metrics_agent(state):

    user_prompt = f"""
Problem:
{state.project.problem}

Objectives:
{state.planning.objectives}
"""

    data = run_gemini(
        METRICS_PROMPT,
        user_prompt
    )

    state.metrics = MetricsOutput(**data)

    return state

In [28]:
RISK_PROMPT = """
You are a Project Risk Consultant.

Identify possible project risks and suggest mitigation strategies.

Return ONLY valid JSON.

{
    "risks":[
        {
            "risk":"",
            "impact":"",
            "probability":"",
            "mitigation":""
        }
    ]
}
"""

In [29]:
def risk_agent(state):

    user_prompt = f"""
Problem:
{state.project.problem}

Project Activities:
{state.planning.activities}
"""

    data = run_gemini(
        RISK_PROMPT,
        user_prompt
    )

    state.risks = RiskOutput(**data)

    return state

In [30]:
FUNDING_PROMPT = """
You are a Grant Funding Expert.

Suggest funding organizations suitable for this project.

Return ONLY valid JSON.

{
    "funding_sources":[
        {
            "organization":"",
            "funding_type":"",
            "reason":"",
            "estimated_amount":""
        }
    ]
}
"""

In [31]:
def funding_agent(state):

    user_prompt = f"""
Problem:
{state.project.problem}

Location:
{state.project.location}

Budget:
{state.project.budget}
"""

    data = run_gemini(
        FUNDING_PROMPT,
        user_prompt
    )

    state.funding = FundingOutput(**data)

    return state

In [32]:
EVALUATION_PROMPT = """
You are an expert Hackathon Judge.

Evaluate the proposed project.

Score it on:
1. Innovation
2. Feasibility
3. Social Impact
4. Sustainability

Return ONLY valid JSON.

{
    "innovation_score":0,
    "feasibility_score":0,
    "impact_score":0,
    "sustainability_score":0,
    "overall_score":0,
    "feedback":""
}
"""

In [33]:
def evaluation_agent(state):

    user_prompt = f"""
Problem:
{state.project.problem}

Objectives:
{state.planning.objectives}

Activities:
{state.planning.activities}

KPIs:
{state.metrics.kpis}
"""

    data = run_gemini(
        EVALUATION_PROMPT,
        user_prompt
    )

    state.evaluation = EvaluationOutput(**data)

    return state

In [34]:
REPORT_PROMPT = """
You are a professional Project Consultant.

Generate a final project report.

Return ONLY valid JSON.

{
    "executive_summary":"",
    "implementation_strategy":"",
    "expected_impact":"",
    "recommendations":""
}
"""

In [35]:
def report_agent(state):

    user_prompt = f"""
Problem:
{state.project.problem}

Research:
{state.research.summary}

Plan:
{state.planning.objectives}

Metrics:
{state.metrics.kpis}

Risks:
{[r.risk for r in state.risks.risks]}

Funding:
{[f.organization for f in state.funding.funding_sources]}
"""

    data = run_gemini(
        REPORT_PROMPT,
        user_prompt
    )

    state.report = ReportOutput(**data)

    return state

In [36]:
def coordinator(state):

    state = research_agent(state)

    state = sdg_agent(state)

    state = planning_agent(state)

    state = metrics_agent(state)

    state = risk_agent(state)

    state = funding_agent(state)

    state = evaluation_agent(state)

    state = report_agent(state)

    return state

In [39]:
state = coordinator(state)

In [40]:
print("="*80)
print("🌍 IMPACTFORGE AI - FINAL PROJECT REPORT")
print("="*80)

print("\n📌 Problem")
print(state.project.problem)

print("\n📖 Research Summary")
print(state.research.summary)

print("\n🌍 Relevant SDGs")
for sdg in state.sdg.sdgs:
    print(f"SDG {sdg.number}: {sdg.title}")

print("\n🎯 Objectives")
for obj in state.planning.objectives:
    print(f"• {obj}")

print("\n📋 Activities")
for activity in state.planning.activities:
    print(f"• {activity}")

print("\n📊 KPIs")
for kpi in state.metrics.kpis:
    print(f"• {kpi}")

print("\n⚠ Risks")
for risk in state.risks.risks:
    print(f"• {risk.risk}")

print("\n💰 Funding Sources")
for fund in state.funding.funding_sources:
    print(f"• {fund.organization}")

print("\n⭐ Evaluation")
print(f"Overall Score : {state.evaluation.overall_score}/100")
print(state.evaluation.feedback)

print("\n📝 Executive Summary")
print(state.report.executive_summary)

print("\n🚀 Expected Impact")
print(state.report.expected_impact)

print("="*80)

🌍 IMPACTFORGE AI - FINAL PROJECT REPORT

📌 Problem
Reduce food waste in Delhi schools

📖 Research Summary
The problem is to significantly reduce food waste generated within school premises in Delhi, specifically focusing on waste originating from school students' meals (both packed and cafeteria-provided). This initiative aims to improve resource efficiency, reduce environmental impact, and foster responsible consumption habits among the younger generation.

🌍 Relevant SDGs
SDG 12: Responsible Consumption and Production
SDG 4: Quality Education
SDG 11: Sustainable Cities and Communities
SDG 13: Climate Action

🎯 Objectives
• Reduce food waste by 30% in participating Delhi schools within 12 months.
• Increase student awareness and active participation in food waste reduction practices by 50% in participating schools.
• Establish sustainable food waste management protocols and infrastructure in 75% of participating schools.

📋 Activities
• Form Project Steering Committee and identify key

# Conclusion

## ImpactForge AI

ImpactForge AI is a multi-agent AI system that helps transform a social problem into a structured, implementation-ready project proposal.

The system consists of specialized AI agents that collaborate to:

- Research the problem
- Identify relevant UN Sustainable Development Goals (SDGs)
- Generate an implementation plan
- Define measurable KPIs
- Assess project risks
- Recommend funding opportunities
- Evaluate project quality
- Produce a professional project report

This modular architecture demonstrates how Google's Gemini models can be orchestrated to support social innovation and accelerate project planning for AI for Good initiatives.

### Future Improvements

- Google ADK Integration
- Retrieval-Augmented Generation (RAG)
- Real-time Government Data APIs
- Interactive Streamlit Dashboard
- Multi-language Support
- Google Maps & Earth Engine Integration
- Vertex AI Deployment